# Interactive Storm Tracks Visualization

This notebook provides an interactive Cartopy map to visualize storm tracks from the PyStormTracker output.

In [ ]:
%matplotlib widget

import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import pandas as pd
import os
from pystormtracker.io.imilast import read_imilast
from pystormtracker.utils.geo import geod_dist_km

In [ ]:
# Load track data
track_file = "../data/test/tracks/era5_msl_2025-2026_djf_2.5x2.5_hodges_imilast.txt"
tracks = read_imilast(track_file)

# Detect track type (VO or MSL)
track_type = 'vo' if 'vo' in os.path.basename(track_file).lower() else 'msl'
print(f"Detected track type: {track_type.upper()}")

In [ ]:
# Convert to Pandas DataFrame
raw_intensity = tracks.vars.get('Intensity1', np.zeros_like(tracks.lats))

data = {
    'track_id': tracks.track_ids,
    'time': tracks.times,
    'lat': tracks.lats,
    'lon': tracks.lons,
    'intensity': raw_intensity
}
df = pd.DataFrame(data)

# Apply scaling factors for user-friendly units
if track_type == 'vo':
    # Scale VO to 10^-5 units
    df['display_intensity'] = df['intensity'] * 1e5
    intensity_unit = "10⁻⁵ s⁻¹"
else:
    # MSL: Convert Pa to hPa
    df['display_intensity'] = df['intensity'] / 100.0
    intensity_unit = "hPa"

# Sort for correct distance/duration calculation
df = df.sort_values(['track_id', 'time'])

In [ ]:
# Pre-calculate track-level statistics
print("Calculating track statistics...")
grouped = df.groupby('track_id')

if track_type == 'vo':
    track_strength = grouped['display_intensity'].max().to_dict()
    strength_desc = f"Min Peak VO ({intensity_unit})"
else:
    track_strength = grouped['display_intensity'].min().to_dict()
    strength_desc = f"Max Peak MSL ({intensity_unit})"
df['track_strength'] = df['track_id'].map(track_strength)

track_durations = (grouped['time'].max() - grouped['time'].min()) / np.timedelta64(1, 'h')
df['track_duration_hrs'] = df['track_id'].map(track_durations.to_dict())

def calc_displacement(g):
    return geod_dist_km(g.iloc[0]['lat'], g.iloc[0]['lon'], g.iloc[-1]['lat'], g.iloc[-1]['lon'])
track_displacements = grouped.apply(calc_displacement)
df['track_displacement_km'] = df['track_id'].map(track_displacements.to_dict())

unique_times = np.sort(df['time'].unique())
print(f"Loaded {len(tracks.track_ids)} points, {len(grouped)} tracks.")

In [ ]:
# Setup User Interface (Plot + Sliders)
import matplotlib.path as mpath
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# 1. Create the Figure
fig = plt.figure(figsize=(10, 8))

# 2. Create the Widgets
proj_dropdown = widgets.Dropdown(
    options=[('Global (Lat/Lon)', 'PlateCarree'), ('Northern Hemisphere', 'NorthPolarStereo'), ('Southern Hemisphere', 'SouthPolarStereo')],
    value='PlateCarree',
    description='Projection:',
    layout={'width': '600px'}
)

time_range_slider = widgets.SelectionRangeSlider(
    options=list(unique_times),
    index=(0, len(unique_times) - 1),
    description='Time Range:',
    layout={'width': '600px'}
)

min_di, max_di = df['display_intensity'].min(), df['display_intensity'].max()
strength_slider = widgets.IntSlider(
    value=int(max_di) if track_type == 'msl' else int(min_di),
    min=int(min_di), max=int(max_di), step=1,
    description=strength_desc, layout={'width': '600px'}
)

duration_slider = widgets.IntSlider(
    value=42, min=0, max=int(df['track_duration_hrs'].max()),
    step=6, description='Min Dur (h):', layout={'width': '600px'}
)

dist_slider = widgets.IntSlider(
    value=1000, min=0, max=int(df['track_displacement_km'].max()),
    step=100, description='Min Dist (km):', layout={'width': '600px'}
)

# Global variables to hold plot state
current_proj = None
ax = None
lc = None
cbar = None

# 3. Update Logic
def update_plot(change):
    global current_proj, ax, lc, cbar
    
    t_start, t_end = time_range_slider.value
    threshold = strength_slider.value
    min_dur = duration_slider.value
    min_dist = dist_slider.value
    proj_type = proj_dropdown.value
    
    # Filter Data
    if track_type == 'vo':
        mask = (df['track_strength'] >= threshold)
    else:
        mask = (df['track_strength'] <= threshold)
    
    mask &= (df['track_duration_hrs'] >= min_dur)
    mask &= (df['track_displacement_km'] >= min_dist)
    
    eligible_tracks = df[mask]['track_id'].unique()
    
    filtered_df = df[
        (df['track_id'].isin(eligible_tracks)) & 
        (df['time'] >= t_start) & 
        (df['time'] <= t_end)
    ]
    
    lines = []
    intensities = []
    if not filtered_df.empty:
        for tid, group in filtered_df.groupby('track_id'):
            if len(group) > 1:
                lons, lats = group['lon'].values, group['lat'].values
                vals = group['display_intensity'].values
                diffs = np.diff(lons)
                splits = np.where(np.abs(diffs) > 180)[0] + 1
                segments_lon = np.split(lons, splits)
                segments_lat = np.split(lats, splits)
                segments_vals = np.split(vals, splits)
                for s_lon, s_lat, s_val in zip(segments_lon, segments_lat, segments_vals):
                    if len(s_lon) > 1:
                        # Create line segments
                        points = np.array([s_lon, s_lat]).T.reshape(-1, 1, 2)
                        segs = np.concatenate([points[:-1], points[1:]], axis=1)
                        lines.extend(segs)
                        # Average intensity for the segment
                        intensities.extend((s_val[:-1] + s_val[1:]) / 2.0)
    
    # Redraw axes if projection changed
    if current_proj != proj_type:
        fig.clf()
        if proj_type == 'NorthPolarStereo':
            crs = ccrs.NorthPolarStereo(central_longitude=0)
        elif proj_type == 'SouthPolarStereo':
            crs = ccrs.SouthPolarStereo(central_longitude=0)
        else:
            crs = ccrs.PlateCarree()
            
        ax = fig.add_subplot(1, 1, 1, projection=crs)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
        ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
        
        if proj_type == 'NorthPolarStereo':
            theta = np.linspace(0, 2*np.pi, 100)
            center, radius = [0.5, 0.5], 0.5
            verts = np.vstack([np.sin(theta), np.cos(theta)]).T
            circle = mpath.Path(verts * radius + center)
            ax.set_boundary(circle, transform=ax.transAxes)
            ax.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
        elif proj_type == 'SouthPolarStereo':
            theta = np.linspace(0, 2*np.pi, 100)
            center, radius = [0.5, 0.5], 0.5
            verts = np.vstack([np.sin(theta), np.cos(theta)]).T
            circle = mpath.Path(verts * radius + center)
            ax.set_boundary(circle, transform=ax.transAxes)
            ax.set_extent([-180, 180, -90, -20], crs=ccrs.PlateCarree())
        else:
            ax.set_global()
            
        cmap = 'turbo' if track_type == 'vo' else 'turbo_r'
        norm = Normalize(vmin=min_di, vmax=max_di)
        lc = LineCollection([], cmap=cmap, norm=norm, linewidths=1.5, alpha=0.8, transform=ccrs.PlateCarree())
        ax.add_collection(lc)
        
        cbar = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax, orientation='vertical', shrink=0.7, pad=0.05)
        cbar.set_label(f'Intensity ({intensity_unit})')
        
        current_proj = proj_type

    lc.set_segments(lines)
    if intensities:
        lc.set_array(np.array(intensities))
    else:
        lc.set_array(np.array([]))
    
    s_str = pd.to_datetime(t_start).strftime('%Y-%m-%d %H:%M')
    e_str = pd.to_datetime(t_end).strftime('%Y-%m-%d %H:%M')
    cond = ">=" if track_type == 'vo' else "<="
    
    ax.set_title(
        f"{track_type.upper()} Tracks: {s_str} to {e_str}\n"
        f"Peak {cond} {threshold} {intensity_unit} | Dur >= {min_dur}h | Dist >= {min_dist}km"
    )
    fig.canvas.draw_idle()

# 4. Display UI and Initial Update
for w in [proj_dropdown, time_range_slider, strength_slider, duration_slider, dist_slider]:
    w.observe(update_plot, names='value')

ui = widgets.VBox([proj_dropdown, time_range_slider, strength_slider, duration_slider, dist_slider])
display(ui)
update_plot(None)
